# 📜 데이터 계약 실습 (Deno + zod)

**커널: Deno** — 이 사이트의 모든 데이터는 `lib/types.ts`(zod) ↔ `pipeline/schemas/*.json` 쌍둥이 계약을 지납니다.
여기서는 **웹이 쓰는 zod 스키마를 그대로 import**해서 계약을 몸으로 익힙니다.

> 파이썬 개발자용 번역: `zod 스키마` ≈ pydantic 모델, `schema.parse(x)` ≈ `Model.model_validate(x)` — 실패하면 예외.

In [1]:
// 셸 헬퍼 — 리포 루트를 찾아 명령을 실행하고 출력을 표시한다
async function findRepo(start = Deno.cwd()): Promise<string> {
  let dir = start;
  while (true) {
    try { await Deno.stat(`${dir}/Makefile`); return dir; } catch { /* 계속 */ }
    const parent = dir.replace(/\/[^/]+$/, "");
    if (parent === dir || parent === "") throw new Error("리포 루트를 못 찾음");
    dir = parent;
  }
}
const REPO = await findRepo();

async function sh(cmd: string): Promise<number> {
  const proc = new Deno.Command("bash", {
    args: ["-lc", cmd], cwd: REPO, stdout: "piped", stderr: "piped",
  });
  const out = await proc.output();
  const text = new TextDecoder().decode(out.stdout) + new TextDecoder().decode(out.stderr);
  console.log(text.trim() || "(출력 없음)");
  return out.code;
}
console.log("리포:", REPO);

리포: /home/rhgw/code/r/r-8282


In [2]:
// 1) 인덱스 읽기 — 어떤 개최일들이 있나
const index = JSON.parse(await Deno.readTextFile(`${REPO}/data/index.json`));
index

{
  schemaVersion: 1,
  updatedAt: "2026-08-16T07:46:57+09:00",
  latestMeetDate: "2026-08-16",
  nextMeetDate: null,
  meetDates: [ "2026-08-16", "2026-08-15", "2026-08-14" ]
}

In [3]:
// 2) 웹의 zod 스키마를 그대로 가져온다 (notes/deno.json의 import map이 "zod"를 npm:zod로 연결)
const types = await import(`file://${REPO}/lib/types.ts`);
const { dataIndexSchema, raceFileSchema } = types;
dataIndexSchema.parse(index);   // 통과하면 조용히 값 반환
"계약 통과 ✓"

"계약 통과 ✓"

In [4]:
// 3) 경주 파일 하나를 파싱해 보기
const date = index.latestMeetDate;
const meet = JSON.parse(await Deno.readTextFile(`${REPO}/data/meets/${date}/meet.json`));
const track = meet.tracks[0];
const raceNo = String(track.races[0].raceNo).padStart(2, "0");
const race = raceFileSchema.parse(JSON.parse(
  await Deno.readTextFile(`${REPO}/data/meets/${date}/${track.track}/r${raceNo}.json`),
));
({ 경주: `${date} ${track.trackName} ${race.raceNo}경주`, 출전: race.entries.length,
   모델: race.prediction?.model, 단승픽: race.prediction?.topPicks.win });

{
  "경주": "2026-08-16 서울 1경주",
  "출전": 9,
  "모델": { statVersion: "v1", aiModel: "haiku" },
  "단승픽": 8
}

In [5]:
// 4) 계약 위반을 일부러 만들어 본다 — 빌드가 왜 '데이터 게이트'인지 체험
const broken = structuredClone(race);
// @ts-ignore 고의 위반
broken.entries[0].sex = "male";   // 허용값은 '수'|'암'|'거'
try {
  raceFileSchema.parse(broken);
} catch (error) {
  console.log(String(error).slice(0, 400));
}
"이 오류가 곧 next build 실패 = 잘못된 데이터가 배포되지 않는 이유"

[
  {
    "code": "invalid_value",
    "values": [
      "수",
      "암",
      "거"
    ],
    "path": [
      "entries",
      0,
      "sex"
    ],
    "message": "Invalid option: expected one of \"수\"|\"암\"|\"거\""
  }
]


"이 오류가 곧 next build 실패 = 잘못된 데이터가 배포되지 않는 이유"

## 규칙 요약 (AGENTS.md 데이터 계약)
- camelCase · 키 생략 금지(`null` 명시) · 날짜/시각은 KST 로컬 문자열 · 기계 타임스탬프만 `+09:00` ISO
- 계약 변경 시 **zod와 JSON Schema 양쪽을 함께** 바꾸고 `schemaVersion` 범프
- `meet.json`·`index.json`은 경주 파일에서 재생성되는 파생물 — 손으로 만지지 않는다